# OpenAI extraction workflow

This template builds a small corpus and extracts structured records with OpenAI text and vision profiles. Set `OPENAI_API_KEY` in the environment before starting Jupyter; never paste it into this notebook. Replace the model placeholder with a model available to your account.

In [ ]:
%env PS_DB=papers.db
%env PS_QUERY=lithium solid electrolyte
%env PS_RECIPE=sse
%env PS_MODEL=YOUR_OPENAI_MODEL
%env PS_OUTPUT=temp_openai_materials.csv
%env PS_FINAL=openai_materials.csv

## Configure model profiles

The same model may be used for both profiles only if it accepts images. Otherwise set separate text and vision identifiers.

In [ ]:
%%bash
set -euo pipefail
test "$PS_MODEL" != "YOUR_OPENAI_MODEL"
ps_model_config text --provider openai --model "$PS_MODEL"
ps_model_config vision --provider openai --model "$PS_MODEL"
ps_model_status

## Build and inspect the corpus

OpenAlex can search without a key. Configure other providers separately if you want broader coverage. Start with a small count, inspect the corpus, and scale only after the complete workflow succeeds.

In [ ]:
%%bash
set -euo pipefail
ps_search "$PS_QUERY" "$PS_DB" --source openalex --count 25
ps_download "$PS_DB" --format both
ps_corpus_stats "$PS_DB"

## Scrape and store

`text-images` uses downloaded text and PDF-derived images. Change the mode to `text` for a cheaper first run. The same recipe is passed to storage so aliases and unit conversions remain consistent.

In [ ]:
%%bash
set -euo pipefail
ps_scrape "$PS_DB" "$PS_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PS_OUTPUT"
ps_store "$PS_DB" "$PS_OUTPUT" "$PS_FINAL" "$PS_RECIPE" --assume-yes
ps_status "$PS_DB"